In [37]:
%pip install --force-reinstall "prettytable==3.11.0"


  Using cached prettytable-3.11.0-py3-none-any.whl.metadata (30 kB)
  Using cached wcwidth-0.9.4-cp310-abi3-macosx_11_0_arm64.whl.metadata (23 kB)
Using cached prettytable-3.11.0-py3-none-any.whl (28 kB)
Using cached wcwidth-0.9.4-cp310-abi3-macosx_11_0_arm64.whl (618 kB)
  Attempting uninstall: wcwidth
    Found existing installation: wcwidth 0.9.4
    Uninstalling wcwidth-0.9.4:
      Successfully uninstalled wcwidth-0.9.4
  Attempting uninstall: prettytable━━━━━━━━━━━━━ 0/2 [wcwidth]
    Found existing installation: prettytable 3.11.032m0/2 [wcwidth]
    Uninstalling prettytable-3.11.0:━━━━━━━━ 0/2 [wcwidth]
      Successfully uninstalled prettytable-3.11.00/2 [wcwidth]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [prettytable]
Note: you may need to restart the kernel to use updated packages.


In [38]:
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server("/tmp/fp_pg", cleanup_mode="stop")

db_url = pg.get_uri()
engine = create_engine(db_url, future=True)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())


PostgreSQL 16.2 on aarch64-apple-darwin23.5.0, compiled by Apple clang version 15.0.0 (clang-1500.0.40.1), 64-bit


In [39]:
%load_ext sql

%sql $db_url


The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [40]:
%config SqlMagic.style = 'DEFAULT'


In [41]:
%%sql

CREATE SCHEMA IF NOT EXISTS pandemic;
SET search_path TO pandemic;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Ffp_pg
Done.
Done.


[]

Прочитай CSV через pandas

In [42]:
import pandas as pd
from pathlib import Path

csv_path = Path("infectious_cases.csv")

df = pd.read_csv(csv_path)

print("Кількість рядків і колонок:", df.shape)
display(df.head())


Кількість рядків і колонок: (10521, 12)


,Entity,Code,Year,Number_yaws,polio_cases,cases_guinea_worm,Number_rabies,Number_malaria,Number_hiv,Number_tuberculosis,Number_smallpox,Number_cholera_cases
0,Afghanistan,AFG,1980,NaN,6160.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,AFG,1981,NaN,5859.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,1982,NaN,9730.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,AFG,1983,NaN,13937.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,AFG,1984,NaN,3864.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [43]:
print(df.columns.tolist())
df.info()


['Entity', 'Code', 'Year', 'Number_yaws', 'polio_cases', 'cases_guinea_worm', 'Number_rabies', 'Number_malaria', 'Number_hiv', 'Number_tuberculosis', 'Number_smallpox', 'Number_cholera_cases']
<class 'pandas.DataFrame'>
RangeIndex: 10521 entries, 0 to 10520
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Entity                10521 non-null  str    
 1   Code                  9393 non-null   str    
 2   Year                  10521 non-null  int64  
 3   Number_yaws           74 non-null     float64
 4   polio_cases           7610 non-null   float64
 5   cases_guinea_worm     8428 non-null   float64
 6   Number_rabies         6780 non-null   float64
 7   Number_malaria        6780 non-null   float64
 8   Number_hiv            6780 non-null   float64
 9   Number_tuberculosis   6780 non-null   float64
 10  Number_smallpox       104 non-null    float64
 11  Number_cholera_cases  2967 non-null   flo

In [44]:
display(df.isna().sum().to_frame("missing_values"))


,missing_values
Entity,0
Code,1128
Year,0
Number_yaws,10447
polio_cases,2911
cases_guinea_worm,2093
Number_rabies,3741
Number_malaria,3741
Number_hiv,3741
Number_tuberculosis,3741


In [45]:
from sqlalchemy import create_engine

engine = create_engine(db_url, future=True)

rows_imported = df.to_sql(
    name="infectious_cases",
    con=engine,
    schema="pandemic",
    if_exists="replace",
    index=False,
    chunksize=1000,
    method="multi",
)

print(f"Імпортовано рядків: {rows_imported}")


Імпортовано рядків: -11


In [46]:
%%sql

SELECT COUNT(*) AS total_rows
FROM pandemic.infectious_cases;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Ffp_pg
1 rows affected.


total_rows
10521


In [47]:
%%sql

SELECT
    column_name,
    data_type,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'pandemic'
  AND table_name = 'infectious_cases'
ORDER BY ordinal_position;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Ffp_pg
12 rows affected.


column_name,data_type,is_nullable
Entity,text,YES
Code,text,YES
Year,bigint,YES
Number_yaws,double precision,YES
polio_cases,double precision,YES
cases_guinea_worm,double precision,YES
Number_rabies,double precision,YES
Number_malaria,double precision,YES
Number_hiv,double precision,YES
Number_tuberculosis,double precision,YES


In [48]:
%%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT "Entity") AS unique_entities,
    MIN("Year") AS min_year,
    MAX("Year") AS max_year,
    COUNT(*) FILTER (
        WHERE "Code" IS NULL
    ) AS missing_codes
FROM pandemic.infectious_cases;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Ffp_pg
1 rows affected.


total_rows,unique_entities,min_year,max_year,missing_codes
10521,245,1920,2023,1128


2. Нормалізуйте таблицю infectious_cases до 3-ї нормальної форми



Створюємо таблицю entities

In [49]:
%%sql

CREATE TABLE IF NOT EXISTS pandemic.entities (
    entity_id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    entity TEXT NOT NULL,
    code TEXT,
    UNIQUE NULLS NOT DISTINCT (entity, code)
);

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Ffp_pg
Done.


[]

In [50]:
%%sql

INSERT INTO pandemic.entities (entity, code)
SELECT DISTINCT
    "Entity",
    "Code"
FROM pandemic.infectious_cases
ON CONFLICT (entity, code) DO NOTHING;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Ffp_pg
0 rows affected.


[]

In [51]:
%%sql

SELECT *
FROM pandemic.entities
ORDER BY entity
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Ffp_pg
10 rows affected.


entity_id,entity,code
212,Afghanistan,AFG
128,Africa,None
77,African Region (WHO),None
102,Albania,ALB
142,Algeria,DZA
56,American Samoa,ASM
20,Americas,None
151,Andorra,AND
221,Angola,AGO
3,Antigua and Barbuda,ATG


Створюємо таблицю з показниками захворювань

In [52]:
%%sql

CREATE TABLE IF NOT EXISTS pandemic.infectious_cases_normalized (
    id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    entity_id INTEGER NOT NULL
        REFERENCES pandemic.entities(entity_id),
    year INTEGER,
    number_yaws DOUBLE PRECISION,
    polio_cases DOUBLE PRECISION,
    cases_guinea_worm DOUBLE PRECISION,
    number_rabies DOUBLE PRECISION,
    number_malaria DOUBLE PRECISION,
    number_hiv DOUBLE PRECISION,
    number_tuberculosis DOUBLE PRECISION,
    number_smallpox DOUBLE PRECISION,
    number_cholera_cases DOUBLE PRECISION
);

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Ffp_pg
Done.


[]

Переносимо дані до нормалізованої таблиці

In [53]:
%%sql

INSERT INTO pandemic.infectious_cases_normalized (
    entity_id,
    year,
    number_yaws,
    polio_cases,
    cases_guinea_worm,
    number_rabies,
    number_malaria,
    number_hiv,
    number_tuberculosis,
    number_smallpox,
    number_cholera_cases
)
SELECT
    e.entity_id,
    i."Year",
    i."Number_yaws",
    i.polio_cases,
    i.cases_guinea_worm,
    i."Number_rabies",
    i."Number_malaria",
    i."Number_hiv",
    i."Number_tuberculosis",
    i."Number_smallpox",
    i."Number_cholera_cases"
FROM pandemic.infectious_cases AS i
JOIN pandemic.entities AS e
    ON i."Entity" = e.entity
   AND i."Code" IS NOT DISTINCT FROM e.code;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Ffp_pg
10521 rows affected.


[]

Перевірка кількості рядків

In [54]:
%%sql

SELECT
    (SELECT COUNT(*)
     FROM pandemic.infectious_cases) AS original_rows,

    (SELECT COUNT(*)
     FROM pandemic.infectious_cases_normalized) AS normalized_rows,

    (SELECT COUNT(*)
     FROM pandemic.entities) AS unique_entities;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Ffp_pg
1 rows affected.


original_rows,normalized_rows,unique_entities
10521,21042,245


Відновлюємо початковий вигляд через JOIN

In [55]:
%%sql

SELECT
    e.entity,
    e.code,
    c.year,
    c.polio_cases,
    c.number_malaria,
    c.number_hiv
FROM pandemic.infectious_cases_normalized AS c
JOIN pandemic.entities AS e
    ON c.entity_id = e.entity_id
ORDER BY e.entity, c.year
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Ffp_pg
10 rows affected.


entity,code,year,polio_cases,number_malaria,number_hiv
Afghanistan,AFG,1960,None,None,None
Afghanistan,AFG,1960,None,None,None
Afghanistan,AFG,1965,None,None,None
Afghanistan,AFG,1965,None,None,None
Afghanistan,AFG,1980,6160.0,None,None
Afghanistan,AFG,1980,6160.0,None,None
Afghanistan,AFG,1981,5859.0,None,None
Afghanistan,AFG,1981,5859.0,None,None
Afghanistan,AFG,1982,9730.0,None,None
Afghanistan,AFG,1982,9730.0,None,None


### Нормалізація таблиці infectious_cases до 3NF

Я нормалізувала початкову таблицю `infectious_cases`, розділивши її на дві пов'язані таблиці: `entities` та `infectious_cases_normalized`.

У таблиці `entities` зберігаються унікальні комбінації назв географічних одиниць (`Entity`) та їхніх кодів (`Code`), кожна з яких має власний первинний ключ `entity_id`.

У таблиці `infectious_cases_normalized` зберігаються роки та статистичні показники захворювань. Замість повторення назв і кодів використовується зовнішній ключ `entity_id`.

Для перенесення даних я використала `INSERT INTO ... SELECT` із `JOIN`. Оскільки в початковому наборі даних є пропущені значення `Code`, застосувала `IS NOT DISTINCT FROM`, щоб коректно обробити `NULL`.

Така нормалізація зменшує дублювання даних, покращує їхню цілісність і дозволяє отримувати повну інформацію через `JOIN`.

### 3. Проаналізуйте дані


In [57]:
%%sql

SELECT
    e.entity_id,
    e.entity,
    e.code,
    ROUND(AVG(c.number_rabies)::NUMERIC, 2) AS avg_rabies,
    MIN(c.number_rabies) AS min_rabies,
    MAX(c.number_rabies) AS max_rabies,
    ROUND(SUM(c.number_rabies)::NUMERIC, 2) AS sum_rabies
FROM pandemic.infectious_cases_normalized AS c
JOIN pandemic.entities AS e
    ON c.entity_id = e.entity_id
GROUP BY
    e.entity_id,
    e.entity,
    e.code
ORDER BY avg_rabies DESC NULLS LAST
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Ffp_pg
10 rows affected.


entity_id,entity,code,avg_rabies,min_rabies,max_rabies,sum_rabies
84,World,OWID_WRL,20192.37,14075.508,24744.658,1211542.22
44,Lower Middle Income (WB),None,15193.96,10202.528,19182.795,911637.56
158,South Asia (WB),None,11729.89,7271.2754,15361.878,703793.33
139,South-East Asia Region (WHO),None,11424.33,6806.0127,15641.958,685459.61
66,G20,None,10189.05,6339.0796,13164.881,611342.76
223,India,IND,8599.17,5425.8726,11121.142,515950.40
119,Sub-Saharan Africa (WB),None,4748.27,4356.8184,5017.977,284896.09
77,African Region (WHO),None,4649.70,4283.073,4917.784,278982.14
193,Low Income (WB),None,3568.11,3014.617,3873.533,214086.33
178,East Asia & Pacific (WB),None,3427.46,2009.9395,4591.0107,205647.67


### Висновок до завдання 3

У цьому завданні я проаналізувала показник `Number_rabies` для кожної унікальної комбінації `Entity` та `Code`, використовуючи нормалізовані таблиці та оператор `JOIN`.

За допомогою агрегатних функцій `AVG()`, `MIN()`, `MAX()` та `SUM()` обчислила середнє, мінімальне, максимальне значення та загальну суму показника. Результати відсортувала за середнім значенням у порядку спадання та обмежила вибірку 10 рядками.

Найвище середнє значення отримано для категорії `World` (20 192.37), далі йдуть `Lower Middle Income (WB)` (15 193.96) та `South Asia (WB)` (11 729.89).

Аналіз також показав, що набір даних містить як окремі країни, так і агреговані регіональні та економічні групи. Тому отриманий рейтинг не можна інтерпретувати виключно як рейтинг країн.

### 4. Побудуйте колонку різниці в роках


In [59]:
%%sql

SELECT DISTINCT
    year,
    MAKE_DATE(year, 1, 1) AS first_january,
    CURRENT_DATE AS current_date,
    EXTRACT(
        YEAR FROM AGE(
            CURRENT_DATE,
            MAKE_DATE(year, 1, 1)
        )
    )::INTEGER AS years_difference
FROM pandemic.infectious_cases_normalized
ORDER BY year
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Ffp_pg
10 rows affected.


year,first_january,current_date,years_difference
1920,1920-01-01,2026-11-08,106
1921,1921-01-01,2026-11-08,105
1922,1922-01-01,2026-11-08,104
1923,1923-01-01,2026-11-08,103
1924,1924-01-01,2026-11-08,102
1925,1925-01-01,2026-11-08,101
1926,1926-01-01,2026-11-08,100
1927,1927-01-01,2026-11-08,99
1928,1928-01-01,2026-11-08,98
1929,1929-01-01,2026-11-08,97


### Висновок до завдання 4

У цьому завданні я використала вбудовані функції PostgreSQL для роботи з датами. За допомогою `MAKE_DATE()` сформувала дату 1 січня для кожного року, `CURRENT_DATE` — отримала поточну дату, а `AGE()` та `EXTRACT()` — обчислила різницю в повних роках.

Додатково використала `DISTINCT`, щоб усунути повторення років у результаті. Наприклад, для 1920 року різниця становить 106 років, а для 1929 року — 97 років станом на 08.11.2026.

У результаті отримала 10 унікальних років із відповідними датами та обчисленою різницею в роках.

### 5. Побудуйте власну функцію

1. Створення функції для розрахунку різниці в роках

In [60]:
%%sql

CREATE OR REPLACE FUNCTION pandemic.calculate_years_difference(
    input_year INTEGER
)
RETURNS INTEGER
LANGUAGE SQL
STABLE
AS $$
    SELECT EXTRACT(
        YEAR FROM AGE(
            CURRENT_DATE,
            MAKE_DATE(input_year, 1, 1)
        )
    )::INTEGER;
$$;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Ffp_pg
Done.


[]

In [61]:
%%sql

SELECT
    1996 AS input_year,
    MAKE_DATE(1996, 1, 1) AS first_january,
    CURRENT_DATE AS current_date,
    pandemic.calculate_years_difference(1996) AS years_difference;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Ffp_pg
1 rows affected.


input_year,first_january,current_date,years_difference
1996,1996-01-01,2026-11-08,30


In [62]:
%%sql

SELECT DISTINCT
    year,
    MAKE_DATE(year, 1, 1) AS first_january,
    CURRENT_DATE AS current_date,
    pandemic.calculate_years_difference(year) AS years_difference
FROM pandemic.infectious_cases_normalized
ORDER BY year
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Ffp_pg
10 rows affected.


year,first_january,current_date,years_difference
1920,1920-01-01,2026-11-08,106
1921,1921-01-01,2026-11-08,105
1922,1922-01-01,2026-11-08,104
1923,1923-01-01,2026-11-08,103
1924,1924-01-01,2026-11-08,102
1925,1925-01-01,2026-11-08,101
1926,1926-01-01,2026-11-08,100
1927,1927-01-01,2026-11-08,99
1928,1928-01-01,2026-11-08,98
1929,1929-01-01,2026-11-08,97


2. Створення функції для середньої кількості захворювань

In [63]:
%%sql

CREATE OR REPLACE FUNCTION pandemic.calculate_avg_cases(
    cases_count DOUBLE PRECISION,
    divisor INTEGER
)
RETURNS NUMERIC
LANGUAGE SQL
IMMUTABLE
AS $$
    SELECT ROUND(
        cases_count::NUMERIC / NULLIF(divisor, 0),
        2
    );
$$;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Ffp_pg
Done.


[]

Перевірка функції на тестових даних

In [64]:
%%sql

SELECT
    pandemic.calculate_avg_cases(1200, 12) AS monthly_avg,
    pandemic.calculate_avg_cases(1200, 4) AS quarterly_avg,
    pandemic.calculate_avg_cases(1200, 2) AS half_year_avg;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Ffp_pg
1 rows affected.


monthly_avg,quarterly_avg,half_year_avg
100.00,300.00,600.00


3. Використання функції з реальною таблицею infectious_cases

In [65]:
%%sql

SELECT
    e.entity,
    e.code,
    c.year,
    c.number_rabies AS annual_cases,

    pandemic.calculate_avg_cases(
        c.number_rabies, 12
    ) AS monthly_avg,

    pandemic.calculate_avg_cases(
        c.number_rabies, 4
    ) AS quarterly_avg,

    pandemic.calculate_avg_cases(
        c.number_rabies, 2
    ) AS half_year_avg

FROM pandemic.infectious_cases_normalized AS c
JOIN pandemic.entities AS e
    ON c.entity_id = e.entity_id

WHERE c.number_rabies IS NOT NULL

ORDER BY e.entity, c.year
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Ffp_pg
10 rows affected.


entity,code,year,annual_cases,monthly_avg,quarterly_avg,half_year_avg
Afghanistan,AFG,1990,2.145772,0.18,0.54,1.07
Afghanistan,AFG,1990,2.145772,0.18,0.54,1.07
Afghanistan,AFG,1991,2.3209748,0.19,0.58,1.16
Afghanistan,AFG,1991,2.3209748,0.19,0.58,1.16
Afghanistan,AFG,1992,2.6180408,0.22,0.65,1.31
Afghanistan,AFG,1992,2.6180408,0.22,0.65,1.31
Afghanistan,AFG,1993,2.8531172,0.24,0.71,1.43
Afghanistan,AFG,1993,2.8531172,0.24,0.71,1.43
Afghanistan,AFG,1994,2.9899871,0.25,0.75,1.49
Afghanistan,AFG,1994,2.9899871,0.25,0.75,1.49


Використання обох функцій в одному запиті

In [66]:
%%sql

SELECT
    e.entity,
    c.year,

    pandemic.calculate_years_difference(
        c.year
    ) AS years_difference,

    c.number_rabies AS annual_cases,

    pandemic.calculate_avg_cases(
        c.number_rabies, 12
    ) AS monthly_avg,

    pandemic.calculate_avg_cases(
        c.number_rabies, 4
    ) AS quarterly_avg,

    pandemic.calculate_avg_cases(
        c.number_rabies, 2
    ) AS half_year_avg

FROM pandemic.infectious_cases_normalized AS c
JOIN pandemic.entities AS e
    ON c.entity_id = e.entity_id

WHERE c.number_rabies IS NOT NULL

ORDER BY e.entity, c.year
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Ffp_pg
10 rows affected.


entity,year,years_difference,annual_cases,monthly_avg,quarterly_avg,half_year_avg
Afghanistan,1990,36,2.145772,0.18,0.54,1.07
Afghanistan,1990,36,2.145772,0.18,0.54,1.07
Afghanistan,1991,35,2.3209748,0.19,0.58,1.16
Afghanistan,1991,35,2.3209748,0.19,0.58,1.16
Afghanistan,1992,34,2.6180408,0.22,0.65,1.31
Afghanistan,1992,34,2.6180408,0.22,0.65,1.31
Afghanistan,1993,33,2.8531172,0.24,0.71,1.43
Afghanistan,1993,33,2.8531172,0.24,0.71,1.43
Afghanistan,1994,32,2.9899871,0.25,0.75,1.49
Afghanistan,1994,32,2.9899871,0.25,0.75,1.49


### Висновок до завдання 5

У цьому завданні я створила дві власні SQL-функції в PostgreSQL.

Перша функція `calculate_years_difference()` приймає рік, формує дату 1 січня за допомогою `MAKE_DATE()` та обчислює різницю в повних роках відносно поточної дати, використовуючи `AGE()` і `EXTRACT()`.

Друга функція `calculate_avg_cases()` приймає кількість випадків захворювання за рік і довільний дільник. Для розрахунку середньої кількості випадків на місяць використовується дільник 12, на квартал — 4, на півріччя — 2. Додатково використано `NULLIF()` для захисту від ділення на нуль.

Обидві функції застосовано до нормалізованої таблиці `infectious_cases_normalized`. Для розрахунків кількості захворювань відібрано лише записи з ненульовими числовими значеннями показника `number_rabies`.

Використання власних SQL-функцій дозволяє повторно застосовувати логіку обчислень у різних запитах, зменшує дублювання коду та спрощує його підтримку.